# 🌡️ Reto Semana 8: MeteoSense Analytics

## Sistema de Análisis de Datos Meteorológicos con NumPy

---

### 📋 Contexto del Proyecto

**MeteoSense** es una startup que instala redes de sensores meteorológicos en ciudades de México para monitorear condiciones ambientales. Has sido contratado como **Analista de Datos Junior** para procesar las mediciones de su red de sensores en la Ciudad de México.

```
╔══════════════════════════════════════════════════════════════════╗
║                    METEOSENSE ANALYTICS                         ║
║                 Red de Monitoreo CDMX 2024                      ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║    📍 Sensores Activos: 5 estaciones                            ║
║    📊 Mediciones: Temperatura, Humedad, CO2                     ║
║    ⏱️  Frecuencia: Lecturas cada hora (24/día)                  ║
║    📅 Período: 7 días de monitoreo                              ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```

### 🎯 Objetivos de Aprendizaje

En este reto aplicarás:
- Creación y manipulación de arrays NumPy
- Indexación y slicing para acceso a datos
- Operaciones vectorizadas para cálculos eficientes
- Funciones estadísticas de NumPy
- Manejo de valores faltantes (NaN)
- Broadcasting para operaciones entre arrays

---

## 📦 Configuración Inicial

Ejecuta esta celda para cargar NumPy y preparar el entorno.

In [ ]:
!pip install numpy
import numpy as np

# Configuración para reproducibilidad
np.random.seed(42)

print("NumPy cargado correctamente")
print(f"   Version: {np.__version__}")


---

## 📊 Datos del Reto

### Estructura de los Datos

```
ESTACIONES DE MONITOREO CDMX
════════════════════════════════════════════

Índice │ Estación          │ Zona
───────┼───────────────────┼──────────────
   0   │ Coyoacán          │ Sur
   1   │ Azcapotzalco      │ Norte
   2   │ Xochimilco        │ Sur
   3   │ Tlalpan           │ Sur
   4   │ Miguel Hidalgo    │ Centro-Oeste

════════════════════════════════════════════

VARIABLES MONITOREADAS
────────────────────────────────────────────
• Temperatura (°C): Rango típico 10-35°C
• Humedad Relativa (%): Rango 20-90%
• CO2 (ppm): Rango típico 350-500 ppm
────────────────────────────────────────────
```

Ejecuta la siguiente celda para generar los datos de los sensores.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                    GENERACIÓN DE DATOS DE SENSORES
# ═══════════════════════════════════════════════════════════════════

estaciones = ['Coyoacán', 'Azcapotzalco', 'Xochimilco', 'Tlalpan', 'Miguel Hidalgo']
n_estaciones = len(estaciones)
n_dias  = 7
n_horas = 24

# ── TEMPERATURA (°C) ─────────────────────────────────────────────────
temp_base = np.array([22, 24, 20, 19, 23])
hora_del_dia    = np.arange(24)
variacion_diaria = 5 * np.sin((hora_del_dia - 6) * np.pi / 12)

temperatura = np.zeros((n_estaciones, n_dias, n_horas))
for i in range(n_estaciones):
    for d in range(n_dias):
        temperatura[i, d, :] = temp_base[i] + variacion_diaria + np.random.normal(0, 1.5, n_horas)

temperatura[1, 2, 10:14] = np.nan   # Azcapotzalco, día 3, horas 10-13
temperatura[3, 5,  0:3]  = np.nan   # Tlalpan, día 6, horas 0-2

# ── HUMEDAD RELATIVA (%) ─────────────────────────────────────────────
humedad_base = np.array([55, 45, 70, 65, 50])
variacion_humedad = -15 * np.sin((hora_del_dia - 6) * np.pi / 12)

humedad = np.zeros((n_estaciones, n_dias, n_horas))
for i in range(n_estaciones):
    for d in range(n_dias):
        humedad[i, d, :] = humedad_base[i] + variacion_humedad + np.random.normal(0, 5, n_horas)

humedad = np.clip(humedad, 20, 95)
humedad[0, 4, 15:18] = np.nan       # Coyoacán, día 5, horas 15-17

# ── CO2 (ppm) ────────────────────────────────────────────────────────
co2_base = np.array([380, 420, 360, 350, 410])
patron_trafico = np.zeros(24)
patron_trafico[7:10]  = 30
patron_trafico[17:20] = 40
patron_trafico[12:14] = 15

co2 = np.zeros((n_estaciones, n_dias, n_horas))
for i in range(n_estaciones):
    for d in range(n_dias):
        co2[i, d, :] = co2_base[i] + patron_trafico + np.random.normal(0, 10, n_horas)

co2[:, 3, :] *= 1.15                # Día de contingencia (día 4)
co2[2, 1, 5:8] = np.nan             # Xochimilco, día 2, horas 5-7

# ── Arrays 2D de promedios diarios ───────────────────────────────────
temp_promedio_diario    = np.nanmean(temperatura, axis=2)   # shape (5, 7)
humedad_promedio_diario = np.nanmean(humedad,     axis=2)
co2_promedio_diario     = np.nanmean(co2,         axis=2)

print("╔══════════════════════════════════════════════════════════════╗")
print("║              DATOS GENERADOS EXITOSAMENTE                    ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  temperatura     : shape {temperatura.shape}               ║")
print(f"║  humedad         : shape {humedad.shape}               ║")
print(f"║  co2             : shape {co2.shape}               ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  temp_promedio_diario    : shape {temp_promedio_diario.shape}            ║")
print(f"║  humedad_promedio_diario : shape {humedad_promedio_diario.shape}            ║")
print(f"║  co2_promedio_diario     : shape {co2_promedio_diario.shape}            ║")
print("╚══════════════════════════════════════════════════════════════╝")
print("\n Estaciones:", estaciones)


---

## 🏋️ PARTE 1: Exploración de Arrays (25 puntos)

### Ejercicio 1.1: Inspección de Datos (5 puntos)

Completa el código para mostrar las propiedades del array `temperatura`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                    EJERCICIO 1.1: INSPECCIÓN
# ═══════════════════════════════════════════════════════════════════

# 1. Número de dimensiones del array temperatura
n_dimensiones = temperatura.ndim

# 2. Forma (shape) del array
forma = temperatura.shape

# 3. Número total de elementos
total_elementos = temperatura.size

# 4. Tipo de datos
tipo_datos = temperatura.dtype

# 5. Tamaño en memoria (bytes)
memoria_bytes = temperatura.nbytes

# Mostrar resultados
print("PROPIEDADES DEL ARRAY TEMPERATURA")
print("─" * 40)
print(f"Dimensiones: {n_dimensiones}D")
print(f"Forma: {forma}")
print(f"  → {forma[0]} estaciones")
print(f"  → {forma[1]} días")
print(f"  → {forma[2]} horas por día")
print(f"Total de mediciones: {total_elementos:,}")
print(f"Tipo de datos: {tipo_datos}")
print(f"Memoria: {memoria_bytes:,} bytes ({memoria_bytes/1024:.2f} KB)")


### Ejercicio 1.2: Indexación Básica (10 puntos)

Usa indexación para extraer datos específicos.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                    EJERCICIO 1.2: INDEXACIÓN
# ═══════════════════════════════════════════════════════════════════

# 1. Temperatura de Coyoacán (índice 0), día 1 (índice 0), a las 12:00 (índice 12)
temp_coyoacan_d1_12h = temperatura[0, 0, 12]
print(f"Coyoacán, Día 1, 12:00h: {temp_coyoacan_d1_12h:.1f}°C")

# 2. Todas las temperaturas de Xochimilco (índice 2) en el día 3 (índice 2)
#    Retorna un array de 24 elementos
temp_xochimilco_d3 = temperatura[2, 2, :]
print(f"\nXochimilco, Día 3 (24 horas):")
print(f"   Primeras 6 horas: {temp_xochimilco_d3[:6].round(1)}")

# 3. Temperatura promedio diario de Miguel Hidalgo (índice 4) para los 7 días
#    Usa el array temp_promedio_diario
temp_mh_7dias = temp_promedio_diario[4, :]
print(f"\nMiguel Hidalgo - Promedio por día:")
print(f"   {temp_mh_7dias.round(1)}")

# 4. Último valor de CO2 registrado (última estación, último día, última hora)
ultimo_co2 = co2[-1, -1, -1]
print(f"\nÚltimo CO2 registrado: {ultimo_co2:.1f} ppm")


### Ejercicio 1.3: Slicing Avanzado (10 puntos)

Usa slicing para extraer subconjuntos de datos.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                    EJERCICIO 1.3: SLICING
# ═══════════════════════════════════════════════════════════════════

# 1. Todas las estaciones, todos los días, horas de tarde (12-17) → shape (5,7,6)
temp_tardes = temperatura[:, :, 12:18]
print(f"Temperaturas de tardes (12-17h)")
print(f"   Shape: {temp_tardes.shape}")

# 2. Primeras 3 estaciones, últimos 3 días, todas las horas → shape (3,3,24)
humedad_subset = humedad[:3, -3:, :]
print(f"\nSubset de humedad")
print(f"   Shape: {humedad_subset.shape}")

# 3. Estaciones pares (0,2,4), todos los días, horas de mañana (6-11) → shape (3,7,6)
co2_mañanas_pares = co2[::2, :, 6:12]
print(f"\nCO2 mañanas (estaciones pares)")
print(f"   Shape: {co2_mañanas_pares.shape}")

# 4. Temperaturas en orden inverso de días → shape (5,7,24)
temp_inverso = temperatura[:, ::-1, :]
print(f"\nTemperatura días invertidos")
print(f"   Shape: {temp_inverso.shape}")


---

## 🏋️ PARTE 2: Estadísticas Básicas (25 puntos)

### Ejercicio 2.1: Estadísticas Globales (10 puntos)

Calcula estadísticas considerando **valores NaN**.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                 EJERCICIO 2.1: ESTADÍSTICAS GLOBALES
# ═══════════════════════════════════════════════════════════════════

# IMPORTANTE: Los arrays tienen valores NaN (sensores desconectados)
# Usamos las funciones nan* de NumPy (nanmean, nanstd, etc.)

# 1. Temperatura promedio global
temp_promedio = np.nanmean(temperatura)

# 2. Temperatura máxima registrada
temp_maxima = np.nanmax(temperatura)

# 3. Temperatura mínima registrada
temp_minima = np.nanmin(temperatura)

# 4. Desviación estándar de temperatura
temp_std = np.nanstd(temperatura)

# 5. Rango de temperatura (máxima - mínima)
temp_rango = temp_maxima - temp_minima

print("╔══════════════════════════════════════════════════════════════╗")
print("║           ESTADÍSTICAS GLOBALES DE TEMPERATURA               ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Promedio:     {temp_promedio:>6.2f} °C                              ║")
print(f"║  Máxima:       {temp_maxima:>6.2f} °C                              ║")
print(f"║  Mínima:       {temp_minima:>6.2f} °C                              ║")
print(f"║  Desv. Est.:   {temp_std:>6.2f} °C                              ║")
print(f"║  Rango:        {temp_rango:>6.2f} °C                              ║")
print("╚══════════════════════════════════════════════════════════════╝")


### Ejercicio 2.2: Estadísticas por Eje (15 puntos)

Calcula estadísticas a lo largo de ejes específicos.

```
RECORDATORIO DE EJES para array 3D (estaciones, días, horas):

       ┌─────────────────────────────────┐
      /│                                /│
     / │        axis=2 (horas)         / │
    /  │        ──────────────►       /  │
   ┌───┼─────────────────────────────┐   │
   │   │                             │   │
   │   │     axis=1 (días)           │   │
   │   │     │                       │   │
   │   │     │                       │   │
   │   │     ▼                       │   │
   │   └─────────────────────────────┼───┘
   │  /                              │  /
   │ / axis=0 (estaciones)           │ /
   │/                                │/
   └─────────────────────────────────┘

• axis=0: Opera sobre estaciones (resultado: días × horas)
• axis=1: Opera sobre días (resultado: estaciones × horas)
• axis=2: Opera sobre horas (resultado: estaciones × días)
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                 EJERCICIO 2.2: ESTADÍSTICAS POR EJE
# ═══════════════════════════════════════════════════════════════════

# 1. Temperatura promedio POR ESTACIÓN → axis=(1,2) → shape (5,)
temp_por_estacion = np.nanmean(temperatura, axis=(1, 2))

print("TEMPERATURA PROMEDIO POR ESTACIÓN")
print("─" * 40)
for i, est in enumerate(estaciones):
    print(f"   {est:15s}: {temp_por_estacion[i]:5.1f} °C")

# 2. Humedad promedio POR HORA DEL DÍA → axis=(0,1) → shape (24,)
humedad_por_hora = np.nanmean(humedad, axis=(0, 1))

# Tabla con bordes +---+ correctamente cerrados
COL1, COL2 = 8, 10
sep = "+" + "-"*COL1 + "+" + "-"*COL2 + "+"
print("\nHUMEDAD PROMEDIO POR HORA")
print(sep)
print(f"| {'Hora':^{COL1-2}} | {'Humedad':^{COL2-2}} |")
print(sep)
for h in [0, 6, 12, 18]:
    print(f"| {f'{h:02d}:00':^{COL1-2}} | {f'{humedad_por_hora[h]:.1f}%':^{COL2-2}} |")
print(sep)

# 3. CO2 máximo POR DÍA → axis=(0,2) → shape (7,)
co2_max_por_dia = np.nanmax(co2, axis=(0, 2))

print("\nCO2 MÁXIMO POR DÍA")
print("─" * 40)
for d in range(n_dias):
    print(f"   Día {d+1}: {co2_max_por_dia[d]:6.1f} ppm")


---

## 🏋️ PARTE 3: Operaciones Vectorizadas (25 puntos)

### Ejercicio 3.1: Conversiones de Unidades (10 puntos)

Realiza conversiones usando operaciones vectorizadas (sin loops).

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#              EJERCICIO 3.1: CONVERSIONES VECTORIZADAS
# ═══════════════════════════════════════════════════════════════════

# Sin loops: operaciones vectorizadas sobre todo el array

# 1. Celsius → Fahrenheit   F = C × 9/5 + 32
temperatura_fahrenheit = temperatura * 9/5 + 32

print("TEMPERATURA EN FAHRENHEIT")
print(f"   Promedio: {np.nanmean(temperatura_fahrenheit):.1f} °F")
print(f"   Máxima:   {np.nanmax(temperatura_fahrenheit):.1f} °F")
print(f"   Mínima:   {np.nanmin(temperatura_fahrenheit):.1f} °F")

# 2. Celsius → Kelvin   K = C + 273.15
temperatura_kelvin = temperatura + 273.15

print(f"\nTEMPERATURA EN KELVIN")
print(f"   Promedio: {np.nanmean(temperatura_kelvin):.1f} K")

# 3. Normalizar humedad a [0, 1]   (valor - min) / (max - min)
humedad_min = np.nanmin(humedad)
humedad_max = np.nanmax(humedad)
humedad_normalizada = (humedad - humedad_min) / (humedad_max - humedad_min)

print(f"\nHUMEDAD NORMALIZADA [0-1]")
print(f"   Promedio: {np.nanmean(humedad_normalizada):.3f}")
print(f"   Min:      {np.nanmin(humedad_normalizada):.3f}")
print(f"   Max:      {np.nanmax(humedad_normalizada):.3f}")


### Ejercicio 3.2: Índice de Confort Térmico (15 puntos)

Calcula un índice simplificado de confort térmico combinando temperatura y humedad.

```
ÍNDICE DE CONFORT TÉRMICO (ICT)
═══════════════════════════════════════════════

Fórmula simplificada:
ICT = T + 0.05 × H

Donde:
  T = Temperatura (°C)
  H = Humedad relativa (%)

Interpretación:
  ICT < 20     → Frío
  20 ≤ ICT < 25 → Confortable
  25 ≤ ICT < 30 → Cálido
  ICT ≥ 30     → Muy caluroso
═══════════════════════════════════════════════
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#              EJERCICIO 3.2: ÍNDICE DE CONFORT TÉRMICO
# ═══════════════════════════════════════════════════════════════════

# 1. ICT = temperatura + 0.05 * humedad  (vectorizado, sin loops)
ict = temperatura + 0.05 * humedad

print("ÍNDICE DE CONFORT TÉRMICO (ICT)")
print("─" * 45)
print(f"   Shape del array ICT: {ict.shape}")
print(f"   ICT promedio: {np.nanmean(ict):.2f}")
print(f"   ICT máximo:   {np.nanmax(ict):.2f}")
print(f"   ICT mínimo:   {np.nanmin(ict):.2f}")

# 2. Clasificar con indexación booleana (sin NaN)
n_frio         = int(np.sum(ict < 20))
n_confortable  = int(np.sum((ict >= 20) & (ict < 25)))
n_calido       = int(np.sum((ict >= 25) & (ict < 30)))
n_muy_caluroso = int(np.sum(ict >= 30))
n_validas      = int(np.sum(~np.isnan(ict)))

# Tabla con tres columnas y bordes +---+ bien cerrados
COL_CAT, COL_CNT, COL_PCT = 24, 6, 8
sep = "+" + "-"*COL_CAT + "+" + "-"*COL_CNT + "+" + "-"*COL_PCT + "+"
filas = [
    ("Frio (<20)",          n_frio,         100*n_frio/n_validas),
    ("Confortable (20-25)", n_confortable,  100*n_confortable/n_validas),
    ("Calido (25-30)",      n_calido,       100*n_calido/n_validas),
    ("Muy caluroso (>=30)", n_muy_caluroso, 100*n_muy_caluroso/n_validas),
]

print("\nDISTRIBUCIÓN DE CONDICIONES")
print(sep)
print(f"| {'Condicion':<{COL_CAT-2}} | {'N':>{COL_CNT-2}} | {'%':>{COL_PCT-2}} |")
print(sep)
for cat, cnt, pct in filas:
    print(f"| {cat:<{COL_CAT-2}} | {cnt:>{COL_CNT-2}} | {pct:>{COL_PCT-3}.1f}% |")
print(sep)
print(f"| {'Total validas':<{COL_CAT-2}} | {n_validas:>{COL_CNT-2}} | {'':>{COL_PCT-2}} |")
print(sep)


---

## 🏋️ PARTE 4: Análisis Avanzado (25 puntos)

### Ejercicio 4.1: Detección de Anomalías (10 puntos)

Identifica valores anómalos usando el método de desviación estándar.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#              EJERCICIO 4.1: DETECCIÓN DE ANOMALÍAS
# ═══════════════════════════════════════════════════════════════════

# Criterio: valor anómalo si está a más de 2 desviaciones estándar de la media

# 1. Media y desviación estándar del CO2
co2_media = np.nanmean(co2)
co2_std   = np.nanstd(co2)

# 2. Límites (media ± 2*std)
limite_inferior = co2_media - 2 * co2_std
limite_superior = co2_media + 2 * co2_std

print("ANÁLISIS DE ANOMALÍAS EN CO2")
print("─" * 45)
print(f"   Media CO2:       {co2_media:.1f} ppm")
print(f"   Desv. Est.:      {co2_std:.1f} ppm")
print(f"   Límite inferior: {limite_inferior:.1f} ppm")
print(f"   Límite superior: {limite_superior:.1f} ppm")

# 3. Máscara booleana: fuera del rango Y no NaN
mascara_anomalias = (~np.isnan(co2)) & ((co2 < limite_inferior) | (co2 > limite_superior))

# 4. Contar anomalías
n_anomalias = int(np.sum(mascara_anomalias))

# 5. Obtener valores anómalos
valores_anomalos = co2[mascara_anomalias]

print(f"\nANOMALÍAS DETECTADAS: {n_anomalias}")
if n_anomalias > 0:
    print(f"   Valores: {valores_anomalos[:10].round(1)}")
    if n_anomalias > 10:
        print(f"   ... y {n_anomalias - 10} más")


### Ejercicio 4.2: Análisis de Contingencia Ambiental (15 puntos)

El día 4 (índice 3) hubo una contingencia ambiental. Analiza cómo afectó las mediciones.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#           EJERCICIO 4.2: ANÁLISIS DE CONTINGENCIA AMBIENTAL
# ═══════════════════════════════════════════════════════════════════

DIA_CONTINGENCIA = 3  # Día 4 (índice 3)

# 1. CO2 del día de contingencia → shape (5, 24)
co2_contingencia = co2[:, DIA_CONTINGENCIA, :]

# 2. CO2 de días normales → shape (5, 6, 24)
dias_normales     = [0, 1, 2, 4, 5, 6]
co2_dias_normales = co2[:, dias_normales, :]

# 3. Promedio CO2 en día de contingencia
promedio_contingencia = np.nanmean(co2_contingencia)

# 4. Promedio CO2 en días normales
promedio_normal = np.nanmean(co2_dias_normales)

# 5. Incremento porcentual: ((contingencia - normal) / normal) * 100
incremento_porcentual = ((promedio_contingencia - promedio_normal) / promedio_normal) * 100

print("╔══════════════════════════════════════════════════════════════╗")
print("║           ANÁLISIS DE CONTINGENCIA AMBIENTAL                 ║")
print("║                        Día 4                                 ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  CO2 promedio día contingencia: {promedio_contingencia:>7.1f} ppm              ║")
print(f"║  CO2 promedio días normales:    {promedio_normal:>7.1f} ppm              ║")
print(f"║  Incremento:                    {incremento_porcentual:>7.1f} %               ║")
print("╚══════════════════════════════════════════════════════════════╝")

# 6. Estación más afectada
co2_por_estacion_contingencia = np.nanmean(co2_contingencia,   axis=1)     # shape (5,)
co2_por_estacion_normal       = np.nanmean(co2_dias_normales,  axis=(1,2)) # shape (5,)

incremento_por_estacion = ((co2_por_estacion_contingencia - co2_por_estacion_normal) /
                            co2_por_estacion_normal) * 100

idx_mas_afectada = np.argmax(incremento_por_estacion)

# Tabla ASCII sin emojis ni bloques Unicode (evita recuadros blancos en terminal)
print("\nIMPACTO POR ESTACIÓN")
print("-" * 50)
for i, est in enumerate(estaciones):
    pct   = incremento_por_estacion[i]
    barra = "#" * int(pct / 2)   # barras ASCII, compatibles con cualquier terminal
    print(f"  {est:15s}: + {pct:.1f}%  {barra}")

print(f"\n  Estación más afectada: {estaciones[idx_mas_afectada]}")


---

## 🎯 EJERCICIO FINAL: Reporte Ejecutivo (BONUS - 10 puntos)

Genera un reporte resumido con las métricas más importantes.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                    EJERCICIO BONUS: REPORTE EJECUTIVO
# ═══════════════════════════════════════════════════════════════════

# 1. Estación más calurosa (mayor temperatura promedio)
idx_mas_calurosa  = np.argmax(np.nanmean(temperatura, axis=(1, 2)))
estacion_mas_calurosa = estaciones[idx_mas_calurosa]

# 2. Estación más húmeda (mayor humedad promedio)
idx_mas_humeda    = np.argmax(np.nanmean(humedad, axis=(1, 2)))
estacion_mas_humeda = estaciones[idx_mas_humeda]

# 3. Estación con mejor calidad de aire (menor CO2 promedio)
idx_mejor_aire    = np.argmin(np.nanmean(co2, axis=(1, 2)))
estacion_mejor_aire = estaciones[idx_mejor_aire]

# 4. Hora más calurosa del día
temp_por_hora     = np.nanmean(temperatura, axis=(0, 1))
hora_mas_calurosa = int(np.argmax(temp_por_hora))

# 5. Hora con peor calidad de aire
co2_por_hora      = np.nanmean(co2, axis=(0, 1))
hora_peor_aire    = int(np.argmax(co2_por_hora))

# 6. Valores faltantes
nan_temperatura = int(np.sum(np.isnan(temperatura)))
nan_humedad     = int(np.sum(np.isnan(humedad)))
nan_co2         = int(np.sum(np.isnan(co2)))
total_nan       = nan_temperatura + nan_humedad + nan_co2

print("")
print("╔══════════════════════════════════════════════════════════════════════╗")
print("║                                                                      ║")
print("║            METEOSENSE - REPORTE EJECUTIVO SEMANAL                   ║")
print("║                  CDMX - Semana de Analisis                          ║")
print("║                                                                      ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print("║                                                                      ║")
print("║  RESUMEN DE CONDICIONES                                              ║")
print("║  ─────────────────────────────────────────────────────────────────   ║")
print(f"║    Temperatura promedio:    {np.nanmean(temperatura):>5.1f} °C                            ║")
print(f"║    Humedad promedio:         {np.nanmean(humedad):>5.1f} %                             ║")
print(f"║    CO2 promedio:            {np.nanmean(co2):>6.1f} ppm                           ║")
print("║                                                                      ║")
print("║  RANKINGS                                                            ║")
print("║  ─────────────────────────────────────────────────────────────────   ║")
print(f"║    Estacion mas calurosa:   {estacion_mas_calurosa:15s}                    ║")
print(f"║    Estacion mas humeda:     {estacion_mas_humeda:15s}                    ║")
print(f"║    Mejor calidad de aire:   {estacion_mejor_aire:15s}                    ║")
print("║                                                                      ║")
print("║  PATRONES TEMPORALES                                                 ║")
print("║  ─────────────────────────────────────────────────────────────────   ║")
print(f"║    Hora mas calurosa:       {hora_mas_calurosa:02d}:00 hrs                              ║")
print(f"║    Hora con mas CO2:         {hora_peor_aire:02d}:00 hrs                              ║")
print("║                                                                      ║")
print("║  CALIDAD DE DATOS                                                    ║")
print("║  ─────────────────────────────────────────────────────────────────   ║")
print(f"║    Valores faltantes totales:  {total_nan:4d}                                   ║")
print(f"║      - Temperatura: {nan_temperatura:3d}                                              ║")
print(f"║      - Humedad:     {nan_humedad:3d}                                              ║")
print(f"║      - CO2:         {nan_co2:3d}                                              ║")
print("║                                                                      ║")
print("╚══════════════════════════════════════════════════════════════════════╝")
print("")


---

## ✅ Checklist de Entrega

Antes de entregar, verifica que:

| # | Criterio | Puntos |
|---|----------|--------|
| 1 | Parte 1: Exploración de Arrays completada | 25 |
| 2 | Parte 2: Estadísticas Básicas completadas | 25 |
| 3 | Parte 3: Operaciones Vectorizadas completadas | 25 |
| 4 | Parte 4: Análisis Avanzado completado | 25 |
| 5 | BONUS: Reporte Ejecutivo completado | +10 |
| | **Total posible** | **110** |

### 📝 Notas Importantes

- **NO uses loops** (for, while) en las partes 2 y 3 - usa operaciones vectorizadas
- **Usa funciones nan*** cuando trabajes con datos que tienen valores faltantes
- **Documenta tu código** si haces algo no obvio
- El código debe ejecutarse sin errores de principio a fin

---

### 🏅 Criterios de Evaluación

```
RÚBRICA DE CALIFICACIÓN
═══════════════════════════════════════════════════════

✓ Código correcto y funcional         → 70%
✓ Uso apropiado de NumPy              → 15%
  (vectorización, funciones nan, ejes)
✓ Resultados razonables               → 10%
✓ Código limpio y organizado          → 5%

═══════════════════════════════════════════════════════
```

---

*MeteoSense Analytics - Reto desarrollado para Programación para Ciencia de Datos, IPN 2026*